In [2]:
import numpy as np
import pandas as pd
from lightgbm import LGBMClassifier
from metric import precision_at_recall

In [3]:
train_features = pd.read_pickle('train_features.pkl')
test_features = pd.read_pickle('test_features.pkl')

In [4]:
train_features.info()

<class 'pandas.DataFrame'>
RangeIndex: 11091 entries, 0 to 11090
Data columns (total 57 columns):
 #   Column                       Non-Null Count  Dtype         
---  ------                       --------------  -----         
 0   cookie_id                    11091 non-null  str           
 1   num_query                    11091 non-null  int64         
 2   pro_seller                   11091 non-null  float64       
 3   platforms                    11091 non-null  str           
 4   platform_count               11091 non-null  int64         
 5   n_events                     11091 non-null  int64         
 6   item_category_count          11091 non-null  int64         
 7   item_id_count                11091 non-null  int64         
 8   item_location_count          11091 non-null  int64         
 9   search_depth                 11091 non-null  float64       
 10  s_page_mean                  11091 non-null  float64       
 11  pointer_null_share           11091 non-null  float64

In [5]:
train_features.head(1)

,cookie_id,num_query,pro_seller,platforms,platform_count,n_events,item_category_count,item_id_count,item_location_count,search_depth,...,max_events_per_minute,no_pointers_cookie,has_enough_for_variation,has_enough_for_search_depth,has_enough_for_pro_seller,has_enough_for_pointer_std,window_start_ts,target,cookie_age,events_per_hour_age
0,ck_000c95f1408dcb00,3,0.083333,desktop+web,2,16,2,9,8,4.0,...,3,0,1,1,1,1,2026-04-19,0,20127782.0,7.949212e-07


In [6]:
test_features.info()

<class 'pandas.DataFrame'>
RangeIndex: 4909 entries, 0 to 4908
Data columns (total 55 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   cookie_id                    4909 non-null   str    
 1   num_query                    4909 non-null   int64  
 2   pro_seller                   4909 non-null   float64
 3   platforms                    4909 non-null   str    
 4   platform_count               4909 non-null   int64  
 5   n_events                     4909 non-null   int64  
 6   item_category_count          4909 non-null   int64  
 7   item_id_count                4909 non-null   int64  
 8   item_location_count          4909 non-null   int64  
 9   search_depth                 4909 non-null   float64
 10  s_page_mean                  4909 non-null   float64
 11  pointer_null_share           4909 non-null   float64
 12  pointer_x_std                4909 non-null   float64
 13  pointer_y_std                

### Baseline
- возьмем константу = ничего не обучаем, ничего не находим, что будет если ничего не делать
- за тем по одному будем добавлять признаки

In [7]:
# возьмем target.mean() - смотрим чисто по  выборке, что дает среднее
cutoff = '2026-04-17' #по аналоогии с quickstart
is_valid = train_features['window_start_ts']>=cutoff


In [8]:
const_score = train_features.loc[~is_valid, 'target'].mean()
y_true = train_features.loc[is_valid, 'target']
score = pd.Series(const_score, index = y_true.index)

In [9]:
baseline_pred = precision_at_recall(y_true, score)
print(baseline_pred)

0.08200922603792926


### BASELINE
- 0.08 = doing nothing
- даже  ниже чем на quickstart, но будем начинать с того, а что будет, если ничего не делать

### LightGBM
- сразу использую его,а не его randomForest
- умеет нативно Nan обрабатывать
- умеет обрабатывать категориальные  признаки без one-hot, categorical_featire=[]
- дерево - не нужна нормализация и масштабирование фичей, инвариантно к монотонным преобразованиям


- n_estimators - число  деревьев
- random_state- для воспроизводства
- min_child_samples  -не дает деревьям переобучаться на маленьких листах
- class_weight

In [10]:
X_train = train_features[~is_valid]
X_val = train_features[is_valid]

y_train = X_train['target']
y_val = X_val['target']


#target - точно убираем
#cookie_id - не пересекаются в train, test и просто идентификатор
#iwindow_start_ts - мы уже сделали is_valid, да и деерво может что-то выучить - чем ближе к концу выборки , тем ...
#               также деревья не уммеют экстраполировать, все тестовые будут лежать в каком-то максимальном window_start_ts > X
#window_end_ts - мы и так не добавляли, оно везде 1 день
X_train = X_train.drop(columns=['target','cookie_id', 'window_start_ts', 'platforms'])
X_val = X_val.drop(columns=['target','cookie_id', 'window_start_ts', 'platforms'])

In [11]:
X_train.columns

Index(['num_query', 'pro_seller', 'platform_count', 'n_events',
       'item_category_count', 'item_id_count', 'item_location_count',
       'search_depth', 's_page_mean', 'pointer_null_share', 'pointer_x_std',
       'pointer_y_std', 'susp_mean', 'n_query', 'median_gap',
       'gaps_share_4_10', 'variation', 'std_gap', 'mean_gap',
       'seller_type_count', 'search_reprise_query', 'contact_chat_open_count',
       'contact_message_sent_count', 'contact_phone_show_count',
       'favorite_add_count', 'item_view_count', 'login_count',
       'photo_swipe_count', 'search_results_view_count',
       'seller_page_view_count', 'contact_chat_open_share',
       'contact_message_sent_share', 'contact_phone_show_share',
       'favorite_add_share', 'item_view_share', 'login_share',
       'photo_swipe_share', 'search_results_view_share',
       'seller_page_view_share', 'events_entropy', 'platform_android',
       'platform_desktop', 'platform_desktop+web', 'platform_ios',
       'platform_w

1 модель
- добавляем признак сначала самый сильный признак median_gap - 0.94 по iv

In [12]:
first_feature_cols = ['median_gap']
first_model = LGBMClassifier(random_state=0)
first_model.fit(X_train[first_feature_cols], y_train)

proba_first = first_model.predict_proba(X_val[first_feature_cols])[:,1] #возращает две колонки : вероятнсоти для класса 0 и класса 1, нам надо второе
first_estemate = precision_at_recall(y_val, proba_first)
first_estemate

[LightGBM] [Info] Number of positive: 739, number of negative: 8401
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000132 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 255
[LightGBM] [Info] Number of data points in the train set: 9140, number of used features: 1


[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.080853 -> initscore=-2.430808
[LightGBM] [Info] Start training from score -2.430808


0.208955223880597

добавляем второй признак
- item_location_count - 0.604 based on ivi

In [13]:
sec_feature_cols =first_feature_cols + ['item_location_count']
sec_model = LGBMClassifier(random_state=0)
sec_model.fit(X_train[sec_feature_cols], y_train)

proba_sec = sec_model.predict_proba(X_val[sec_feature_cols])[:,1] 
sec_estemate = precision_at_recall(y_val, proba_sec)
sec_estemate

[LightGBM] [Info] Number of positive: 739, number of negative: 8401
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000157 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 286
[LightGBM] [Info] Number of data points in the train set: 9140, number of used features: 2
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.080853 -> initscore=-2.430808
[LightGBM] [Info] Start training from score -2.430808


0.198943661971831

добавляем третий признак
- search_depth 0.542 - ivi

In [14]:
third_feature_cols =sec_feature_cols + ['search_depth']
third_model = LGBMClassifier(random_state=0)
third_model.fit(X_train[third_feature_cols], y_train)

proba_third = third_model.predict_proba(X_val[third_feature_cols])[:,1] 
third_estemate = precision_at_recall(y_val, proba_third)
third_estemate

[LightGBM] [Info] Number of positive: 739, number of negative: 8401
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000117 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 323
[LightGBM] [Info] Number of data points in the train set: 9140, number of used features: 3
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.080853 -> initscore=-2.430808
[LightGBM] [Info] Start training from score -2.430808


0.22396856581532418

добавляем четвертый признак
- variation - из той же ветки, что и median_gap, но корреляция была не высока
- имеет 0.505

In [15]:
forth_feature_cols =third_feature_cols + ['variation']
forth_model = LGBMClassifier(random_state=0)
forth_model.fit(X_train[forth_feature_cols], y_train)

proba_forth = forth_model.predict_proba(X_val[forth_feature_cols])[:,1] 
forth_estemate = precision_at_recall(y_val, proba_forth)
forth_estemate

[LightGBM] [Info] Number of positive: 739, number of negative: 8401
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000360 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 578
[LightGBM] [Info] Number of data points in the train set: 9140, number of used features: 4
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.080853 -> initscore=-2.430808
[LightGBM] [Info] Start training from score -2.430808


0.19430485762144054

добавляем пятый признак
- item_category_counnt 0.364

In [16]:
fifth_feature_cols =forth_feature_cols + ['item_category_count']
fifth_model = LGBMClassifier(random_state=0)
fifth_model.fit(X_train[fifth_feature_cols], y_train)

proba_fifth = fifth_model.predict_proba(X_val[fifth_feature_cols])[:,1] 
fifth_estemate = precision_at_recall(y_val, proba_fifth)
fifth_estemate

[LightGBM] [Info] Number of positive: 739, number of negative: 8401
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000075 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 587
[LightGBM] [Info] Number of data points in the train set: 9140, number of used features: 5
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.080853 -> initscore=-2.430808
[LightGBM] [Info] Start training from score -2.430808


0.2912371134020619

добавляем шестой признак
- gaps_share_4_10 - тоже из ветки time_gap, корреляция была низкая - проверим
- 0.421 iv

In [17]:
sixth_feature_cols =fifth_feature_cols + ['gaps_share_4_10']
sixth_model = LGBMClassifier(random_state=0)
sixth_model.fit(X_train[sixth_feature_cols], y_train)

proba_sixth = sixth_model.predict_proba(X_val[sixth_feature_cols])[:,1] 
sixth_estemate = precision_at_recall(y_val, proba_sixth)
sixth_estemate
#стало даже хуже, поэтому дальше будем идти от 5 модели, 
# но проверим сначала убрав variation
#median убирать глупо, самый влиятельный признак

[LightGBM] [Info] Number of positive: 739, number of negative: 8401
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000406 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 842
[LightGBM] [Info] Number of data points in the train set: 9140, number of used features: 6
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.080853 -> initscore=-2.430808
[LightGBM] [Info] Start training from score -2.430808


0.2909090909090909

седьмая модель
- стало даже хуже, поэтому дальше будем идти от 5 модели, 
- но проверим сначала убрав variation
- median убирать глупо, самый влиятельный признак

In [18]:
seventh_feature_cols =third_feature_cols + ['gaps_share_4_10'] + ['item_category_count']
seventh_model = LGBMClassifier(random_state=0)
seventh_model.fit(X_train[seventh_feature_cols], y_train)

proba_seventh = seventh_model.predict_proba(X_val[seventh_feature_cols])[:,1] 
seventh_estemate = precision_at_recall(y_val, proba_seventh)
seventh_estemate
#все таки оставляем variation и идем от шестой модели

[LightGBM] [Info] Number of positive: 739, number of negative: 8401
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000169 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 587
[LightGBM] [Info] Number of data points in the train set: 9140, number of used features: 5
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.080853 -> initscore=-2.430808
[LightGBM] [Info] Start training from score -2.430808


0.2889447236180904

восьмая модель - идем от пятой модели
- добавляем признак cookie_age - 0.289 iv
- пошли довольно низкие показетели по iv

In [19]:
eigth_feature_cols =fifth_feature_cols + ['cookie_age']
eigth_model = LGBMClassifier(random_state=0)
eigth_model.fit(X_train[eigth_feature_cols], y_train)

proba_eigth = eigth_model.predict_proba(X_val[eigth_feature_cols])[:,1] 
eigth_estemate = precision_at_recall(y_val, proba_eigth)
eigth_estemate
#пока что тоже не удачная очень

[LightGBM] [Info] Number of positive: 739, number of negative: 8401
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000181 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 842
[LightGBM] [Info] Number of data points in the train set: 9140, number of used features: 6
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.080853 -> initscore=-2.430808
[LightGBM] [Info] Start training from score -2.430808


0.3192090395480226

девятая модель - от пятой
- pgoto_swipe_share - 0.284

In [20]:
ninth_feature_cols =fifth_feature_cols + ['photo_swipe_share']
ninth_model = LGBMClassifier(random_state=0)
ninth_model.fit(X_train[ninth_feature_cols], y_train)

proba_ninth = ninth_model.predict_proba(X_val[ninth_feature_cols])[:,1] 
ninth_estemate = precision_at_recall(y_val, proba_ninth)
ninth_estemate

[LightGBM] [Info] Number of positive: 739, number of negative: 8401
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000159 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 800
[LightGBM] [Info] Number of data points in the train set: 9140, number of used features: 6
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.080853 -> initscore=-2.430808
[LightGBM] [Info] Start training from score -2.430808


0.3027027027027027

десятая модель - от пятой
- favourite_add_share - 0.235

In [21]:
tenth_feature_cols =fifth_feature_cols + ['favorite_add_share']
tenth_model = LGBMClassifier(random_state=0)
tenth_model.fit(X_train[tenth_feature_cols], y_train)

proba_tenth = tenth_model.predict_proba(X_val[tenth_feature_cols])[:,1] 
tenth_estemate = precision_at_recall(y_val, proba_tenth)
tenth_estemate

[LightGBM] [Info] Number of positive: 739, number of negative: 8401
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000264 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 786
[LightGBM] [Info] Number of data points in the train set: 9140, number of used features: 6
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.080853 -> initscore=-2.430808
[LightGBM] [Info] Start training from score -2.430808


0.29815303430079154

одинадцатая - от девятой
 - тот же признак favourite

In [22]:
elevtnh_feature_cols =ninth_feature_cols + ['favorite_add_share']
elevnth_model = LGBMClassifier(random_state=0)
elevnth_model.fit(X_train[elevtnh_feature_cols], y_train)

proba_ele = elevnth_model.predict_proba(X_val[elevtnh_feature_cols])[:,1] 
ele_estemate = precision_at_recall(y_val, proba_ele)
ele_estemate

[LightGBM] [Info] Number of positive: 739, number of negative: 8401
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000489 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 999
[LightGBM] [Info] Number of data points in the train set: 9140, number of used features: 7
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.080853 -> initscore=-2.430808
[LightGBM] [Info] Start training from score -2.430808


0.3210227272727273

двенадцатая - от десятой модели
- pro_seller - 0.153 ivi

In [23]:
twelveth_feature_cols =tenth_feature_cols + ['pro_seller']
twelveth_model = LGBMClassifier(random_state=0)
twelveth_model.fit(X_train[twelveth_feature_cols], y_train)

proba_twl = twelveth_model.predict_proba(X_val[twelveth_feature_cols])[:,1] 
twl_estemate = precision_at_recall(y_val, proba_twl)
twl_estemate

[LightGBM] [Info] Number of positive: 739, number of negative: 8401
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000276 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 984
[LightGBM] [Info] Number of data points in the train set: 9140, number of used features: 7
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.080853 -> initscore=-2.430808
[LightGBM] [Info] Start training from score -2.430808


0.2962962962962963

двенадцатая с половиной - pro_seller + has_enough_for_seller
- вметсе с флагом

In [24]:
twelveth_feature_cols_half =tenth_feature_cols + ['pro_seller' , 'has_enough_for_pro_seller']
twelveth_model_half = LGBMClassifier(random_state=0)
twelveth_model_half.fit(X_train[twelveth_feature_cols_half], y_train)

proba_twl_half = twelveth_model_half.predict_proba(X_val[twelveth_feature_cols_half])[:,1] 
twl_half_estemate = precision_at_recall(y_true, proba_twl_half)
twl_half_estemate

[LightGBM] [Info] Number of positive: 739, number of negative: 8401
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000297 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 986
[LightGBM] [Info] Number of data points in the train set: 9140, number of used features: 8
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.080853 -> initscore=-2.430808
[LightGBM] [Info] Start training from score -2.430808


0.29015544041450775

тринадцая - от десятой модели
- num_query- 0.143 iv

In [25]:
thierteens_feature_cols =tenth_feature_cols + ['num_query']
thrt_model = LGBMClassifier(random_state=0)
thrt_model.fit(X_train[thierteens_feature_cols], y_train)

proba_thrt = thrt_model.predict_proba(X_val[thierteens_feature_cols])[:,1] 
trt_estemate = precision_at_recall(y_val, proba_thrt)
trt_estemate

[LightGBM] [Info] Number of positive: 739, number of negative: 8401
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000209 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 809
[LightGBM] [Info] Number of data points in the train set: 9140, number of used features: 7
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.080853 -> initscore=-2.430808
[LightGBM] [Info] Start training from score -2.430808


0.2864450127877238

четырнадцатая модель - от десятой
- susp_mean без (platoform dummies)

In [26]:
fortn_feature_cols =tenth_feature_cols + ['susp_mean']
frtn_model = LGBMClassifier(random_state=0)
frtn_model.fit(X_train[fortn_feature_cols], y_train)

proba_frtn = frtn_model.predict_proba(X_val[fortn_feature_cols])[:,1] 
frtn_estemate = precision_at_recall(y_val, proba_frtn)
frtn_estemate

[LightGBM] [Info] Number of positive: 739, number of negative: 8401
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000238 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 788
[LightGBM] [Info] Number of data points in the train set: 9140, number of used features: 7
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.080853 -> initscore=-2.430808
[LightGBM] [Info] Start training from score -2.430808


0.2986666666666667

In [27]:
fortn_feature_cols_half =fortn_feature_cols + ['platform_android', 'platform_desktop', 'platform_desktop+web', 'platform_ios', 'platform_web']
frtn_model_half = LGBMClassifier(random_state=0)
frtn_model_half.fit(X_train[fortn_feature_cols_half], y_train)

proba_frtn_half = frtn_model_half.predict_proba(X_val[fortn_feature_cols_half])[:,1] 
frtn_estemate_half = precision_at_recall(y_val, proba_frtn_half)
frtn_estemate_half

[LightGBM] [Info] Number of positive: 739, number of negative: 8401
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000289 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 798
[LightGBM] [Info] Number of data points in the train set: 9140, number of used features: 12
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.080853 -> initscore=-2.430808
[LightGBM] [Info] Start training from score -2.430808


0.2864450127877238

пятнадцатая модель - от 15
- pointer_null_share (которой тоже нужен dummy)

In [28]:
fiftn_feature_cols =tenth_feature_cols + ['pointer_null_share']
fiftn_model = LGBMClassifier(random_state=0)
fiftn_model.fit(X_train[fiftn_feature_cols], y_train)

proba_fiftn = fiftn_model.predict_proba(X_val[fiftn_feature_cols])[:,1] 
fiftn_estemate = precision_at_recall(y_val, proba_fiftn)
fiftn_estemate

[LightGBM] [Info] Number of positive: 739, number of negative: 8401
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000333 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 899
[LightGBM] [Info] Number of data points in the train set: 9140, number of used features: 7
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.080853 -> initscore=-2.430808
[LightGBM] [Info] Start training from score -2.430808


0.3284457478005865

In [29]:
fiftn_feature_cols_half =fiftn_feature_cols + ['platform_android', 'platform_desktop', 'platform_desktop+web', 'platform_ios', 'platform_web']
fiftn_model_half = LGBMClassifier(random_state=0)
fiftn_model_half.fit(X_train[fiftn_feature_cols_half], y_train)

proba_fiftn_half = fiftn_model_half.predict_proba(X_val[fiftn_feature_cols_half])[:,1] 
fiftn_estemate_half = precision_at_recall(y_val, proba_fiftn_half)
fiftn_estemate_half

[LightGBM] [Info] Number of positive: 739, number of negative: 8401
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000451 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 909
[LightGBM] [Info] Number of data points in the train set: 9140, number of used features: 12
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.080853 -> initscore=-2.430808
[LightGBM] [Info] Start training from score -2.430808


0.3137254901960784

16 - от десятой модели
- добавим флаг has_enough_for_search_depth

In [30]:
sixtn_feature_cols =tenth_feature_cols + ['has_enough_for_search_depth']
sixtn_model = LGBMClassifier(random_state=0)
sixtn_model.fit(X_train[sixtn_feature_cols], y_train)

proba_sixth = sixtn_model.predict_proba(X_val[sixtn_feature_cols])[:,1] 
sixtn_estemate = precision_at_recall(y_val, proba_sixth)
sixtn_estemate

[LightGBM] [Info] Number of positive: 739, number of negative: 8401
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000098 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 788
[LightGBM] [Info] Number of data points in the train set: 9140, number of used features: 7
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.080853 -> initscore=-2.430808
[LightGBM] [Info] Start training from score -2.430808


0.29815303430079154

17 модель - от 10 
- n_events

In [31]:
s17_feature_cols =tenth_feature_cols + ['n_events']
s17_model = LGBMClassifier(random_state=0)
s17_model.fit(X_train[s17_feature_cols], y_train)

proba_s17 = s17_model.predict_proba(X_val[s17_feature_cols])[:,1] 
s17_estemate = precision_at_recall(y_val, proba_s17)
s17_estemate

[LightGBM] [Info] Number of positive: 739, number of negative: 8401
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000382 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 896
[LightGBM] [Info] Number of data points in the train set: 9140, number of used features: 7
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.080853 -> initscore=-2.430808
[LightGBM] [Info] Start training from score -2.430808


0.2955145118733509

18 model - from 17
- бинарная версия no_pointers_cookie

In [32]:
s18_feature_cols =s17_feature_cols + ['no_pointers_cookie']
s18_model = LGBMClassifier(random_state=0)
s18_model.fit(X_train[s18_feature_cols], y_train)

proba_s18 = s18_model.predict_proba(X_val[s18_feature_cols])[:,1] 
s18_estemate = precision_at_recall(y_val, proba_s18)
s18_estemate

[LightGBM] [Info] Number of positive: 739, number of negative: 8401
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000458 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 898
[LightGBM] [Info] Number of data points in the train set: 9140, number of used features: 8
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.080853 -> initscore=-2.430808
[LightGBM] [Info] Start training from score -2.430808


0.32748538011695905

19 model - from 17
- has_enough_for_variation - flag

In [33]:
s19_feature_cols =s17_feature_cols + ['has_enough_for_variation']
s19_model = LGBMClassifier(random_state=0)
s19_model.fit(X_train[s19_feature_cols], y_train)

proba_s19 = s19_model.predict_proba(X_val[s19_feature_cols])[:,1] 
s19_estemate = precision_at_recall(y_val, proba_s19)
s19_estemate

[LightGBM] [Info] Number of positive: 739, number of negative: 8401
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000100 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 898
[LightGBM] [Info] Number of data points in the train set: 9140, number of used features: 8
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.080853 -> initscore=-2.430808
[LightGBM] [Info] Start training from score -2.430808


0.2955145118733509

In [34]:
features = s17_feature_cols

### Добавление новых признаков
- после получения низкого скора
- pointer_std_x,y
- s_page_mean -> пробелма, что есть search_depth
- events_per_hour_age вместо n_events
- в любую уж можно вставить events_entropy (добавим под конец в 17, и в новую ветку)
- ветка новая пойдет от 10, а к 17 просто добавим entropy

In [35]:
s20_model = LGBMClassifier(random_state=0)
s20_features_cols = tenth_feature_cols + ['pointer_x_std']
s20_model.fit(X_train[s20_features_cols], y_train)

proba_s20 = s20_model.predict_proba(X_val[s20_features_cols])[:,1] 
s20_estemate = precision_at_recall(y_val, proba_s20)
s20_estemate 
#ОЧЕНЬ НЕПЛОХО

[LightGBM] [Info] Number of positive: 739, number of negative: 8401
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000176 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1041
[LightGBM] [Info] Number of data points in the train set: 9140, number of used features: 7
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.080853 -> initscore=-2.430808
[LightGBM] [Info] Start training from score -2.430808


0.5233644859813084

pointer_x_std - НАХОДКА

In [36]:
s21_model = LGBMClassifier(random_state=0)
s21_features_cols = s20_features_cols + ['pointer_y_std']
s21_model.fit(X_train[s21_features_cols], y_train)

proba_s21 = s21_model.predict_proba(X_val[s21_features_cols])[:,1] 
s21_estemate = precision_at_recall(y_val, proba_s21)
s21_estemate 
#есть прирост, оставляем

[LightGBM] [Info] Number of positive: 739, number of negative: 8401
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000229 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1296
[LightGBM] [Info] Number of data points in the train set: 9140, number of used features: 8
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.080853 -> initscore=-2.430808
[LightGBM] [Info] Start training from score -2.430808


0.5358851674641149

In [37]:
s21_features_cols

['median_gap',
 'item_location_count',
 'search_depth',
 'variation',
 'item_category_count',
 'favorite_add_share',
 'pointer_x_std',
 'pointer_y_std']

pointer_y_std - не так хорошо, но прирост есть, добавляем

In [38]:
s22_features_cols = ['favorite_add_share',
 'item_category_count',
 'item_location_count',
 'median_gap',
 'pointer_x_std',
 'pointer_y_std',
#  'search_depth', - замена на s_page_mean
 'variation'] + ['s_page_mean']
s22_features_cols
#тут вот есть favourite_add_share - будет ли мешать 
#кода будем добавлять events_entropy

['favorite_add_share',
 'item_category_count',
 'item_location_count',
 'median_gap',
 'pointer_x_std',
 'pointer_y_std',
 'variation',
 's_page_mean']

In [39]:
s22_model = LGBMClassifier(random_state=0)
s22_model.fit(X_train[s22_features_cols], y_train)

proba_s22 = s22_model.predict_proba(X_val[s22_features_cols])[:,1] 
s22_estemate = precision_at_recall(y_val, proba_s22)
s22_estemate 
# небольшими шагами, но все же идем!!!!!

[LightGBM] [Info] Number of positive: 739, number of negative: 8401
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000233 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1509
[LightGBM] [Info] Number of data points in the train set: 9140, number of used features: 8
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.080853 -> initscore=-2.430808
[LightGBM] [Info] Start training from score -2.430808


0.509009009009009

In [40]:
s23_model = LGBMClassifier(random_state=0)
s23_features_cols = s22_features_cols + ['events_entropy']
s23_model.fit(X_train[s23_features_cols], y_train)


proba_s23 = s23_model.predict_proba(X_val[s23_features_cols])[:,1] 
s23_estemate = precision_at_recall(y_val, proba_s23)
s23_estemate

[LightGBM] [Info] Number of positive: 739, number of negative: 8401
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000964 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1764
[LightGBM] [Info] Number of data points in the train set: 9140, number of used features: 9
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.080853 -> initscore=-2.430808
[LightGBM] [Info] Start training from score -2.430808


0.5572139303482587

In [41]:
s24_features_cols = [
    # 'favorite_add_share', #попроюуем убрать
 'item_category_count',
 'item_location_count',
 'median_gap',
 'pointer_x_std',
 'pointer_y_std',
 'variation',
 's_page_mean',
 'events_entropy'
]

In [42]:
s24_model = LGBMClassifier(random_state=0)
s24_model.fit(X_train[s24_features_cols], y_train)


proba_s24 = s24_model.predict_proba(X_val[s24_features_cols])[:,1] 
s24_estemate = precision_at_recall(y_val, proba_s24)
s24_estemate
#красота

[LightGBM] [Info] Number of positive: 739, number of negative: 8401
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.002107 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1565
[LightGBM] [Info] Number of data points in the train set: 9140, number of used features: 8
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.080853 -> initscore=-2.430808
[LightGBM] [Info] Start training from score -2.430808


0.5022421524663677

In [43]:
s25_model = LGBMClassifier(random_state=0)
s25_features_cols = s24_features_cols + ['events_per_hour_age']
s25_model.fit(X_train[s25_features_cols], y_train)


proba_s25 = s25_model.predict_proba(X_val[s25_features_cols])[:,1] 
s25_estemate = precision_at_recall(y_val, proba_s25)
s25_estemate


[LightGBM] [Info] Number of positive: 739, number of negative: 8401
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000545 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1820
[LightGBM] [Info] Number of data points in the train set: 9140, number of used features: 9
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.080853 -> initscore=-2.430808
[LightGBM] [Info] Start training from score -2.430808


0.5714285714285714

#### lightGBM
- они терпимее к коррелирующим признакам, чем линейные модели, может быть стоить добавить несколько признаков (mean, std) к одной ветви мысли?
- может старые признаки заиграют?

In [44]:
s26_model = LGBMClassifier(random_state=0)
s26_features_cols = s24_features_cols + ['no_pointers_cookie']
s26_model.fit(X_train[s26_features_cols], y_train)

proba_s26 = s26_model.predict_proba(X_val[s26_features_cols])[:,1]
s26_estemate = precision_at_recall(y_val, proba_s26)
s26_estemate
#нет


[LightGBM] [Info] Number of positive: 739, number of negative: 8401
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000175 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1567
[LightGBM] [Info] Number of data points in the train set: 9140, number of used features: 9
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.080853 -> initscore=-2.430808
[LightGBM] [Info] Start training from score -2.430808


0.5

In [45]:
s27_model = LGBMClassifier(random_state=0)
s27_features_cols = s24_features_cols + ['susp_mean']
s27_model.fit(X_train[s27_features_cols], y_train)

proba_s27 = s27_model.predict_proba(X_val[s27_features_cols])[:,1]
s27_estemate = precision_at_recall(y_val, proba_s27)
s27_estemate
# не такой провал, но тоже плохо

[LightGBM] [Info] Number of positive: 739, number of negative: 8401
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000251 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1567
[LightGBM] [Info] Number of data points in the train set: 9140, number of used features: 9
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.080853 -> initscore=-2.430808
[LightGBM] [Info] Start training from score -2.430808


0.509090909090909

In [46]:
#добавила еще std_gap для анализа активности
s28_model = LGBMClassifier(random_state=0)
s28_features_cols = s24_features_cols + ['std_gap']
s28_model.fit(X_train[s28_features_cols], y_train)

proba_s28 = s28_model.predict_proba(X_val[s28_features_cols])[:,1]
s28_estemate = precision_at_recall(y_val, proba_s28)
s28_estemate
# не помогло

[LightGBM] [Info] Number of positive: 739, number of negative: 8401
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000392 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1820
[LightGBM] [Info] Number of data points in the train set: 9140, number of used features: 9
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.080853 -> initscore=-2.430808
[LightGBM] [Info] Start training from score -2.430808


0.4808510638297872

### Так как очень большие различия между этим тестом и на степике
- надо бы переписать чутка тестовую проверку
- можно изменить cutoffs, 

In [47]:
cutoffs = ['2026-04-14', '2026-04-15', '2026-04-16', '2026-04-17', '2026-04-18']
def check_across_cutoffs(features, params):
    scores = []
    for cutoff in cutoffs:
        is_valid_c = train_features['window_start_ts'] >= cutoff
        Xtr_c, ytr_c = train_features.loc[~is_valid_c, features], train_features.loc[~is_valid_c, 'target']
        Xva_c, yva_c = train_features.loc[is_valid_c, features], train_features.loc[is_valid_c, 'target']
        model = LGBMClassifier(**params)
        model.fit(Xtr_c, ytr_c)
        proba = model.predict_proba(Xva_c)[:, 1]
        scores.append(precision_at_recall(yva_c, proba))
    scores = np.array(scores)
    print(f"mean={scores.mean():.4f} std={scores.std():.4f} min={scores.min():.4f} max={scores.max():.4f}")
    print([round(s, 3) for s in scores])
    return scores


In [48]:

check_across_cutoffs(s24_features_cols, dict(random_state=0, verbose=-1))
#ну похуже, конечно, но результат уже должен стать лучше

mean=0.5218 std=0.0375 min=0.4620 max=0.5722
[np.float64(0.544), np.float64(0.572), np.float64(0.529), np.float64(0.502), np.float64(0.462)]


array([0.54366812, 0.57223796, 0.52881356, 0.50224215, 0.46202532])

#### Еще одна стратегия
- раньше добавлялись по одному признаки, смотрелось как они единично влияют на изменения
- можно закинуть побольше параметров,  с теми же mean, media, std - и регуляризацию добавить 
- теперь проверка будет группами!

In [49]:
FEATURE_GROUPS = [
    ('gap-extra', ['mean_gap', 'std_gap']),
    ('event-shares-rest', ['item_view_share', 'search_results_view_share',
                            'seller_page_view_share', 'contact_phone_show_share',
                            'login_share', 'contact_chat_open_share', 'contact_message_sent_share']),
    ('event-counts', ['item_view_count', 'search_results_view_count', 'photo_swipe_count',
                       'favorite_add_count', 'seller_page_view_count', 'contact_phone_show_count',
                       'login_count', 'contact_chat_open_count', 'contact_message_sent_count']),
    ('listing-extra', ['item_id_count', 'seller_type_count', 'platform_count']),
    ('search-extra', ['n_query', 'search_reprise_query']),
    ('burst', ['max_events_per_minute']),
]

current_cols = s24_features_cols.copy()
current_mean = check_across_cutoffs(current_cols, dict(random_state=0, verbose=-1, num_leaves=15)).mean()

log = [{'group': 'model s24', 'mean_before': None, 'mean_after': current_mean, 'decision': ''}]

for name, cols in FEATURE_GROUPS:
    trial_cols = current_cols + cols
    trial_scores = check_across_cutoffs(trial_cols, dict(random_state=0, verbose=-1, num_leaves=15))
    trial_mean = trial_scores.mean()
    keep = trial_mean > current_mean
    log.append({
        'group': name,
        'mean_before': current_mean,
        'mean_after': trial_mean,
        'decision': 'keep' if keep else 'remove',
    })
    if keep:
        current_cols = trial_cols
        current_mean = trial_mean

ablation_log = pd.DataFrame(log)
ablation_log

mean=0.5650 std=0.0193 min=0.5410 max=0.5969
[np.float64(0.55), np.float64(0.57), np.float64(0.597), np.float64(0.568), np.float64(0.541)]
mean=0.5685 std=0.0291 min=0.5156 max=0.5923
[np.float64(0.589), np.float64(0.587), np.float64(0.592), np.float64(0.558), np.float64(0.516)]
mean=0.6436 std=0.0127 min=0.6286 max=0.6667
[np.float64(0.64), np.float64(0.638), np.float64(0.644), np.float64(0.667), np.float64(0.629)]
mean=0.6355 std=0.0203 min=0.6061 max=0.6550
[np.float64(0.616), np.float64(0.606), np.float64(0.653), np.float64(0.655), np.float64(0.648)]
mean=0.6518 std=0.0256 min=0.6128 max=0.6788
[np.float64(0.632), np.float64(0.613), np.float64(0.675), np.float64(0.679), np.float64(0.66)]
mean=0.6777 std=0.0365 min=0.6401 max=0.7320
[np.float64(0.647), np.float64(0.64), np.float64(0.71), np.float64(0.732), np.float64(0.66)]
mean=0.6675 std=0.0196 min=0.6385 max=0.6957
[np.float64(0.638), np.float64(0.676), np.float64(0.654), np.float64(0.696), np.float64(0.673)]


,group,mean_before,mean_after,decision
0,model s24,NaN,0.565038,
1,gap-extra,0.565038,0.568519,keep
2,event-shares-rest,0.568519,0.643558,keep
3,event-counts,0.643558,0.635506,remove
4,listing-extra,0.643558,0.651802,keep
5,search-extra,0.651802,0.677717,keep
6,burst,0.677717,0.667516,remove


In [50]:
param_grid_5 = [
    {},
    {'num_leaves': 15},
    {'min_child_samples': 30},
    {'num_leaves': 15, 'min_child_samples': 30},
    {'n_estimators': 50},
    {'class_weight': 'balanced'},
]

for params in param_grid_5:
    full_params = dict(random_state=0, verbose=-1, **params)
    print(params)
    check_across_cutoffs(current_cols, full_params)
    print()

{}
mean=0.6466 std=0.0379 min=0.5776 max=0.6860
[np.float64(0.65), np.float64(0.686), np.float64(0.675), np.float64(0.644), np.float64(0.578)]

{'num_leaves': 15}
mean=0.6777 std=0.0365 min=0.6401 max=0.7320
[np.float64(0.647), np.float64(0.64), np.float64(0.71), np.float64(0.732), np.float64(0.66)]

{'min_child_samples': 30}
mean=0.6345 std=0.0404 min=0.5641 max=0.6788
[np.float64(0.647), np.float64(0.619), np.float64(0.664), np.float64(0.679), np.float64(0.564)]

{'num_leaves': 15, 'min_child_samples': 30}
mean=0.6811 std=0.0232 min=0.6579 max=0.7097
[np.float64(0.662), np.float64(0.658), np.float64(0.71), np.float64(0.709), np.float64(0.667)]

{'n_estimators': 50}
mean=0.6537 std=0.0219 min=0.6147 max=0.6754
[np.float64(0.654), np.float64(0.673), np.float64(0.675), np.float64(0.651), np.float64(0.615)]

{'class_weight': 'balanced'}
mean=0.6401 std=0.0438 min=0.5877 max=0.7099
[np.float64(0.637), np.float64(0.602), np.float64(0.664), np.float64(0.71), np.float64(0.588)]



In [51]:

final_model = LGBMClassifier(random_state=0, verbose=-1, num_leaves=15, min_child_samples=30)
final_model.fit(train_features[current_cols], train_features['target'])

test_scores = final_model.predict_proba(test_features[current_cols])[:, 1]

submission = pd.DataFrame({
    'cookie_id': test_features['cookie_id'],
    'score': test_scores,
})
submission.to_csv('submission_check.csv', index=False)

In [52]:
# current_cols